# Hospitality Analytics Dashboard
Interactive KPI summary for Revenue & Occupancy, Guest Satisfaction, and Loyalty Performance.

In [ ]:
# Load gold tables
df_scorecard = spark.read.format('delta').table('gold_property_scorecards').limit(100000).toPandas()
df_channel   = spark.read.format('delta').table('gold_channel_performance').limit(100000).toPandas()
df_loyalty   = spark.read.format('delta').table('gold_loyalty_segments').limit(100000).toPandas()
df_weekly    = spark.read.format('delta').table('gold_weekly_trends').limit(100000).toPandas()
df_occupancy = spark.read.format('delta').table('gold_occupancy_analysis').limit(100000).toPandas()

In [ ]:
total_revenue   = round(df_scorecard['total_revenue'].sum(), 0)
total_bookings  = int(df_scorecard['total_bookings'].sum())
avg_revpar      = round(df_scorecard['revpar'].mean(), 2)
avg_adr         = round(df_scorecard['adr'].mean(), 2)
avg_satisfaction = round(df_scorecard['avg_overall_score'].mean(), 1)
avg_los         = round(df_scorecard['avg_los'].mean(), 1)

print(f"Total Revenue      : ${total_revenue:,.0f}")
print(f"Total Bookings     : {total_bookings:,}")
print(f"Average RevPAR     : ${avg_revpar:,.2f}")
print(f"Average ADR        : ${avg_adr:,.2f}")
print(f"Avg Satisfaction   : {avg_satisfaction}/10")
print(f"Avg Length of Stay : {avg_los} nights")

In [ ]:
# Build top-10 properties by RevPAR
top_props = (
    df_scorecard[['property_name', 'city', 'star_rating', 'total_bookings', 'adr', 'revpar', 'avg_overall_score', 'performance_band']]
    .sort_values('revpar', ascending=False)
    .head(10)
)

# Build channel summary
channel_rows = df_channel[['channel', 'total_bookings', 'total_revenue', 'avg_rate', 'conversion_rate']].sort_values('total_revenue', ascending=False)

html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Hospitality Analytics Dashboard</title>
<style>
  body {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1   {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2   {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:160px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; margin-bottom:30px; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Excellent        {{ background:#dff6dd; color:#107c10; }}
  .Good             {{ background:#cceaff; color:#0078d4; }}
  .Needs.Improvement{{ background:#fff4ce; color:#b7610e; }}
  .Critical         {{ background:#fde7e9; color:#a80000; }}
</style></head><body>
<h1>🏨 Hospitality Analytics Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">${total_revenue:,.0f}</div><div class="lbl">Total Revenue</div></div>
  <div class="kpi"><div class="val">{total_bookings:,}</div><div class="lbl">Total Bookings</div></div>
  <div class="kpi"><div class="val">${avg_revpar}</div><div class="lbl">Avg RevPAR</div></div>
  <div class="kpi"><div class="val">${avg_adr}</div><div class="lbl">Avg ADR</div></div>
  <div class="kpi"><div class="val">{avg_satisfaction}/10</div><div class="lbl">Guest Satisfaction</div></div>
  <div class="kpi"><div class="val">{avg_los}n</div><div class="lbl">Avg Length of Stay</div></div>
</div>

<h2>Top 10 Properties by RevPAR</h2>
<table>
  <tr><th>Property</th><th>City</th><th>Stars</th><th>Bookings</th><th>ADR</th><th>RevPAR</th><th>Satisfaction</th><th>Performance</th></tr>
"""
for _, r in top_props.iterrows():
    band = str(r['performance_band']).replace(' ', '.')
    html += f"""
  <tr>
    <td>{r['property_name']}</td>
    <td>{r['city']}</td>
    <td>{'⭐' * int(r['star_rating'])}</td>
    <td>{int(r['total_bookings']):,}</td>
    <td>${r['adr']:,.2f}</td>
    <td>${r['revpar']:,.2f}</td>
    <td>{r['avg_overall_score']}/10</td>
    <td><span class="badge {band}">{r['performance_band']}</span></td>
  </tr>"""

html += """
</table>

<h2>Booking Channel Performance</h2>
<table>
  <tr><th>Channel</th><th>Bookings</th><th>Revenue</th><th>Avg Rate</th><th>Conversion Rate</th></tr>
"""
for _, r in channel_rows.iterrows():
    html += f"""
  <tr>
    <td>{r['channel']}</td>
    <td>{int(r['total_bookings']):,}</td>
    <td>${r['total_revenue']:,.2f}</td>
    <td>${r['avg_rate']:,.2f}</td>
    <td>{r['conversion_rate']}%</td>
  </tr>"""

html += """
</table>
</body></html>"""

displayHTML(html)

In [ ]:
# Save dashboard as HTML artifact
with open('/lakehouse/default/Files/hospitality_dashboard.html', 'w') as f:
    f.write(html)
print('Dashboard saved to lakehouse Files/hospitality_dashboard.html')